# Part 2: Results Analysis & Discussion
Notebook này trình bày kết quả phân tích và so sánh mô hình trên bộ dữ liệu thực.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from part1.ols_implementation import ols_fit, coef_inference, vif
from part1.ridge_lasso import ridge_fit

from data_pipeline import DataPipeline, load_data, train_test_split
from model_comparison import run_diagnostics, train_models, hyperparameter_tuning, comparison_table, plot_coefficients
from advanced_methods import kernel_ridge_fit

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 2. EDA khảo sát và chẩn đoán

### 2.1. Tổng quan Dữ liệu gốc

In [ ]:
data_path = Path("data/melb_data.csv")
df = load_data(data_path)
print("=== Xem trước các dòng đầu của bộ dữ liệu gốc ===")
display(df.head())

print("=== Khảo sát sơ bộ tổng thể ===")
display(df.info())

Schema xác nhận `Price` là target, dữ liệu gồm numeric và categorical;

### 2.2. Phân tách Dữ liệu thô

In [ ]:
X_train_raw, X_test_raw = train_test_split(df, test_size=0.3, random_state=42)

split_summary = pd.DataFrame(
    {
        "split": ["X_train_raw", "X_test_raw"],
        "rows": [len(X_train_raw), len(X_test_raw)],
        "pct": [round(len(X_train_raw) / len(df) * 100, 2), round(len(X_test_raw) / len(df) * 100, 2)],
    }
)
display(split_summary)

Split được thực hiện trên DataFrame thô. Các thống kê dùng để chọn preprocessing chính thức lấy từ `X_train_raw`.


### 2.3. Thống kê mô tả: mean, median, std, min, max, quartiles

In [ ]:
numeric_raw_columns = df.select_dtypes(include=np.number).columns.tolist()
raw_descriptive_stats = pd.DataFrame(
    {
        "mean": df[numeric_raw_columns].mean(),
        "median": df[numeric_raw_columns].median(),
        "std": df[numeric_raw_columns].std(),
        "min": df[numeric_raw_columns].min(),
        "q1": df[numeric_raw_columns].quantile(0.25),
        "q2": df[numeric_raw_columns].quantile(0.50),
        "q3": df[numeric_raw_columns].quantile(0.75),
        "max": df[numeric_raw_columns].max(),
    }
).round(2)
raw_descriptive_stats

Bảng thống kê raw dùng để nhìn biên độ và độ lệch giữa mean/median.


In [ ]:
duplicate_rows = int(df.duplicated().sum())
duplicate_check = pd.DataFrame(
    {
        "metric": ["duplicate_rows", "duplicate_pct"],
        "value": [duplicate_rows, round(duplicate_rows / len(df) * 100, 2)],
    }
)
duplicate_check

Duplicate check chỉ kiểm tra dòng trùng hoàn toàn trên raw dataset; pipeline baseline không drop row ở bước EDA.


### 2.4. Phân tích Missing Values

In [ ]:
train_missing_summary = pd.DataFrame(
    {
        "missing_count": X_train_raw.isna().sum(),
        "missing_pct": X_train_raw.isna().mean().mul(100).round(2),
    }
).query("missing_count > 0").sort_values("missing_pct", ascending=False)
train_missing_summary

`BuildingArea` và `YearBuilt` thiếu nhiều nhất, nên dùng median train-only thay vì mean để giảm ảnh hưởng của phân phối lệch và outlier.


### 2.5. Phát hiện Outliers & Invalid Values

In [ ]:
train_invalid_summary = pd.DataFrame(
    [
        {"check": "Price <= 0", "rows": int((X_train_raw["Price"] <= 0).sum())},
        {"check": "Rooms == 0", "rows": int((X_train_raw["Rooms"] == 0).sum())},
        {"check": "Landsize <= 0", "rows": int((X_train_raw["Landsize"] <= 0).sum())},
        {"check": "BuildingArea <= 0", "rows": int((X_train_raw["BuildingArea"] <= 0).sum())},
        {"check": "YearBuilt < 1800", "rows": int((X_train_raw["YearBuilt"] < 1800).sum())},
    ]
)
train_invalid_summary

Các điều kiện invalid này có ý nghĩa theo miền dữ liệu nên được repair trong pipeline; outlier giá trị lớn không bị drop tự động.


In [ ]:
outlier_columns = ["Price", "Landsize", "BuildingArea", "YearBuilt"]
train_outlier_quantiles = X_train_raw[outlier_columns].quantile([0.01, 0.05, 0.50, 0.95, 0.99]).T.round(2)
train_outlier_quantiles

In [ ]:
iqr_rows = []
for column in outlier_columns:
    series = X_train_raw[column].dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = int(((series < lower_bound) | (series > upper_bound)).sum())
    iqr_rows.append(
        {
            "column": column,
            "iqr_outliers": outlier_count,
            "outlier_pct_iqr": round(outlier_count / len(series) * 100, 2),
            "iqr_lower_bound": round(lower_bound, 2),
            "iqr_upper_bound": round(upper_bound, 2),
        }
    )

train_iqr_outlier_summary = pd.DataFrame(iqr_rows).sort_values("outlier_pct_iqr", ascending=False)
train_iqr_outlier_summary

Quantile và IQR tập trung vào `Price`, `Landsize`, `BuildingArea`, `YearBuilt` vì đây là các biến ảnh hưởng trực tiếp tới imputation, scaling và feature engineering. Không dùng z-score vì các biến này lệch phải mạnh.


### 2.6. Ma trận Tương quan & Biểu đồ

In [ ]:
plot_columns = ["Price", "Landsize", "BuildingArea", "Distance", "YearBuilt"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

for ax, column in zip(axes, plot_columns):
    sns.histplot(data=X_train_raw, x=column, bins=40, ax=ax)
    ax.set_title(column)

for ax in axes[len(plot_columns):]:
    ax.set_visible(False)

plt.tight_layout()

In [ ]:
corr_columns = ["Price", "Rooms", "Bedroom2", "Bathroom", "Distance", "BuildingArea", "YearBuilt"]
focused_corr = X_train_raw[corr_columns].corr(numeric_only=True)
plt.figure(figsize=(8, 6))
sns.heatmap(focused_corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", linewidths=0.5)
plt.title("Focused Train Correlation Heatmap")
plt.tight_layout()


Heatmap tập trung vào các biến có rủi ro đa cộng tuyến hoặc liên quan trực tiếp tới `Price`. `Bedroom2` chồng lấp với `Rooms`, còn `YearBuilt` được chuyển thành `Age` rồi drop trong pipeline.


In [ ]:
plot_columns = ["Price", "Landsize", "BuildingArea", "Distance", "YearBuilt"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

for ax, column in zip(axes, plot_columns):
    sns.boxplot(data=X_train_raw, y=column, ax=ax)
    ax.set_title(column)

for ax in axes[len(plot_columns):]:
    ax.set_visible(False)

plt.tight_layout()

Boxplot dùng để nhận diện đuôi dài và điểm cực trị của các biến chính; pipeline chỉ repair invalid values, không loại bỏ nhà giá cao hoặc đất lớn hợp lệ.


In [ ]:
numeric_columns = [
    "Price",
    "Rooms",
    "Bedroom2",
    "Bathroom",
    "Car",
    "Distance",
    "Landsize",
    "BuildingArea",
    "YearBuilt",
    "Propertycount",
]

numeric_profile = X_train_raw[numeric_columns].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
numeric_profile["skew"] = X_train_raw[numeric_columns].skew(numeric_only=True)
numeric_profile.round(2)


`Landsize`, `BuildingArea`, `Distance` có scale lệch mạnh so với biến đếm; scaling sau preprocessing giúp ma trận OLS ổn định hơn.


`Price`, `Landsize`, `BuildingArea` lệch phải nên các bước fill và fallback ưu tiên median.


In [ ]:
relationship_columns = ["Price", "Rooms", "Bedroom2", "Bathroom", "Car", "Distance", "BuildingArea", "YearBuilt"]
relationship_summary = X_train_raw[relationship_columns].corr(numeric_only=True)[["Price", "Rooms"]].round(3)
relationship_summary


Bảng tương quan nhỏ giữ lại phần số cụ thể cho quyết định drop `Bedroom2` và không tạo `OtherRooms`, tránh phụ thuộc tuyến tính không cần thiết cho OLS.


In [ ]:
categorical_columns = ["Type", "Regionname", "Method", "CouncilArea"]
categorical_summary = pd.DataFrame(
    {
        "column": categorical_columns,
        "unique_values": [X_train_raw[column].nunique(dropna=True) for column in categorical_columns],
        "missing_pct": [round(X_train_raw[column].isna().mean() * 100, 2) for column in categorical_columns],
        "top_value": [X_train_raw[column].mode(dropna=True).iloc[0] for column in categorical_columns],
    }
)
categorical_summary


In [ ]:
type_price = X_train_raw.groupby("Type")["Price"].agg(["count", "median", "mean"]).round(0)
region_price = (
    X_train_raw.groupby("Regionname")["Price"]
    .agg(["count", "median"])
    .sort_values("median", ascending=False)
    .round(0)
)

type_price, region_price


Với baseline OLS, `Type` và `Regionname` đủ gọn để one-hot mà vẫn giữ tín hiệu loại nhà và khu vực. Bảng median price theo nhóm giải thích vì sao hai cột này được giữ.


In [ ]:
geo_df = X_train_raw[X_train_raw["Price"] < X_train_raw["Price"].quantile(0.99)]
plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    geo_df["Longtitude"],
    geo_df["Lattitude"],
    c=geo_df["Price"],
    alpha=0.35,
    s=10,
)
plt.xlabel("Longtitude")
plt.ylabel("Lattitude")
plt.title("Price by location, bottom 99%")
plt.colorbar(scatter, label="Price")
plt.tight_layout()


Geo scatter chứng minh `Lattitude` và `Longtitude` có tín hiệu không gian rõ. Lọc bottom 99% chỉ để nhìn plot dễ hơn, không thay đổi dữ liệu pipeline.


In [ ]:
feature_preview = X_train_raw[["Rooms", "BuildingArea", "Landsize", "YearBuilt", "Date"]].copy()
feature_preview["SaleYear"] = pd.to_datetime(feature_preview["Date"], dayfirst=True, errors="coerce").dt.year
feature_preview["Age"] = feature_preview["SaleYear"] - feature_preview["YearBuilt"]
feature_preview["BuildingArea_per_Room"] = feature_preview["BuildingArea"] / feature_preview["Rooms"].replace(0, np.nan)
feature_preview["BuildingCoverage"] = feature_preview["BuildingArea"] / feature_preview["Landsize"].replace(0, np.nan)
feature_preview[["Age", "BuildingArea_per_Room", "BuildingCoverage"]].describe().T.round(2)


`Age` dùng năm bán từ `Date`, không dùng mốc năm hiện tại. Ratio features giữ ở mức đơn giản và được fallback bằng median train nếu phát sinh NaN/inf.


## Phase 1: Vòng lặp xử lí dữ liệu gốc

### 1. Tiền xử lý dữ liệu

In [ ]:
# TODO: Preprocessing pipeline
# Khởi tạo và chạy pipeline trên tập train thô

# --- QUÁ TRÌNH KHÁM BỆNH VÀ LẶP (Chạy nháp nhiều lần để ra kết quả) ---
# Vòng 0: Mảng rỗng [] -> Chạy VIF -> DA phát hiện Bedroom2 bị lỗi
# Vòng 1: Mảng ['Bedroom2'] -> Chạy VIF -> DA phát hiện thêm Type_t bị lỗi
# Vòng 2: Mảng ['Bedroom2', 'Type_t'] -> Chạy VIF -> Sạch hoàn toàn!

current_drop = ["Bedroom2", "Type_t"]
pipeline_loop = DataPipeline(drop_columns=current_drop)
X_train_temp, y_train_temp = pipeline_loop.fit_transform(df_train_raw)

### 2. Chẩn đoán sơ khởi

In [ ]:
diagnostics_df = run_diagnostics(
    X_train_raw=X_train_temp,
    y_train=y_train_temp,
    feature_names=pipeline_loop.encoded_columns,
    custom_ols_func=ols_fit,
    custom_vif_func=vif,
    custom_inference_func=coef_inference,
)

# Hiển thị bảng kết quả cho Data Analyst xem xét và ra quyết định
diagnostics_df.sort_values(by="VIF", ascending=False).head(10)

### Sync point: Đánh giá & quyết định
* **Nhận xét của Data Analyst:** Dựa vào bảng chẩn đoán trên, các biến `Rooms` và `Bedroom2` có chỉ số $VIF > 10$, đồng thời hệ số p-value của `Bedroom2` lớn hơn 0.05 (không có ý nghĩa thống kê).
* **Quyết định trảm biến:** Nhóm thống nhất loại bỏ cột `Bedroom2` ra khỏi hệ thống để làm sạch ma trận đặc trưng.

## Phase 2: Huấn luyện & Đánh giá mô hình

### 1. Tiền xử lí dữ liệu cho mô hình

In [ ]:
# Khởi tạo Pipeline mới nhận lệnh loại bỏ cột từ Data Analyst
final_drop_list = ["Bedroom2", "Type_t"]  # Example

# 1. Pipeline Baseline (Cấu hình rỗng [] - Giữ nguyên 21 cột gốc)
pipeline_baseline = DataPipeline(drop_columns=[])
X_train_raw, y_train = pipeline_baseline.fit_transform(df_train_raw)
X_test_raw, _ = pipeline_baseline.transform(df_test_raw)

# 2. Pipeline Tối ưu (Cấu hình final_drop_list - Đã lọc sạch đa cộng tuyến)
pipeline_best = DataPipeline(drop_columns=final_drop_list)
X_train_best, y_train = pipeline_best.fit_transform(df_train_raw)  # Ghi đè y_train an toàn
X_test_best, y_test = pipeline_best.transform(df_test_raw)

# 3. Đóng gói Metadata bàn giao (Contract 1)
metadata = {
    "feature_names": pipeline_best.encoded_columns,
    "target_name": "Price",
    "pipeline": pipeline_best,
    "imputation_strategy": "knn_imputer",
    "scaling_method": "standardize",
}

### 2. Tìm Siêu Tham Số Ridge Qua K-Fold CV

In [ ]:
# ML Modeler tiến hành dò tìm thông số phạt tốt nhất trên tập X_train_best
lambda_grid = [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0]
from model_comparison import custom_ridge_fit  # Hàm lõi phần 1

best_lambda_ridge, cv_scores = hyperparameter_tuning(
    X_train=X_train_best,
    y_train=y_train,
    model_class=ridge_fit,
    param_grid={"alpha": lambda_grid},
    k=5,
)
print(f"Hệ số phạt tối ưu cho Ridge tìm được: \u03bb = {best_lambda_ridge}")

### 3. Huấn luyện mô hình

In [ ]:
# 6. Chạy 4 mô hình song song hoàn toàn bằng code tự chế (Mở két sắt chứa tập Test để chấm điểm)
results_dict = train_models(
    X_train_raw=X_train_raw,
    X_train_best=X_train_best,
    y_train=y_train,
    X_test_raw=X_test_raw,
    X_test_best=X_test_best,
    y_test=y_test,
    custom_ols_func=ols_fit,
    custom_ridge_func=ridge_fit,
    custom_kernel_func=kernel_ridge_fit,  # Nhận hàm core Toán phi tuyến từ Member C
    lambda_ridge=best_lambda_ridge,
    lambda_kernel=1.0,  # Giả định lambda cho mô hình Kernel nâng cao
)

### 4. DataFrame

In [ ]:
# ML Modeler chuyển giao bảng xếp hạng hiệu năng tổng hợp cho Data Analyst
df_leaderboard = comparison_table(results_dict)
df_leaderboard

## Phase 3: Đánh giá phần dư và trực quan hóa
* **Mục tiêu:** Vẽ biểu đồ chẩn đoán sức khỏe phần dư của mô hình vô địch, trực quan hóa mức độ quan trọng của đặc trưng nhằm viết báo cáo Typst thực tế.

In [ ]:
# 1. Vẽ biểu đồ hệ số thanh ngang giải thích tầm quan trọng của các biến sạch
plot_coefficients(results=results_dict, feature_names=metadata["feature_names"])

# 2. Trích xuất phần dư của mô hình tốt nhất để phân tích kiểm định Gauss-Markov
from model_comparison import evaluate_gauss_markov_assumptions, plot_predictions

best_model_key = "Ridge_custom"
best_residuals = y_test - results_dict[best_model_key]["predictions_test"]

gm_metrics = evaluate_gauss_markov_assumptions(X_train_best, y_train, best_residuals)
plot_predictions(y_test, results_dict)

## Phase 4: Biện luận & Tổng kết báo cáo (Cell này nên bỏ, tập trung viết bên Typst)
*(Data Analyst sử dụng các kết quả định lượng trên cell để viết báo cáo khoa học)*
1. **Lý do Ridge vượt trội OLS:** Giải thích dựa trên việc bóp nghẹt phương sai của các biến tương quan cao.
2. **Ý nghĩa thực tế của hệ số hồi quy:** Phân tích các biến động vị trí ảnh hưởng trực tiếp thế nào tới giá nhà đất tại Melbourne.